In [1]:
import json
# load alignment_results.json
with open("alignment_results.json", "r") as file:
    alignment_results = json.load(file)

In [2]:
alignment_results

{'A1': {'Theme': 'Space & Humanity: The human aspiration to explore and inhabit space.',
  'Entity': ['Hannah Reid: An advocate for mental health awareness and support.',
   'Helix Labs: At the forefront of genetic research and innovation.',
   'Science Fair Extravaganza: A festival celebrating innovations and discoveries.',
   "Lunar Mission Launch: An event marking mankind's leap into space exploration.",
   'Professor Ivan Chen: A pioneer in space exploration technologies.'],
  'Time': ['2019-01-01', '2020-08-02'],
  'Prev': []},
 'B1': {'Theme': 'Frontiers of Knowledge: Breakthroughs and discoveries that redefine our understanding.',
  'Entity': ['Science Fair Extravaganza: A festival celebrating innovations and discoveries.',
   'Professor Ivan Chen: A pioneer in space exploration technologies.'],
  'Time': ['2019-01-01', '2020-08-02'],
  'Prev': []},
 'B2': {'Theme': 'Frontiers of Knowledge: Breakthroughs and discoveries that redefine our understanding.',
  'Entity': ['Science Fa

In [5]:
from langchain.llms import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from typing import Union, List
from langchain.output_parsers import PydanticOutputParser
from meta.my_types import Chapter, Story

def generate_chapter_prompt(
    time_period, themes, entity, prev_content, format_instructions
):
    return f"""
    You are a senior reporter at the New York Times and you are writing a fictional news story that continues from a previous story. Your writing style is concise and crisp, without rhetoric or superfluous narration. The continuation chapter needs to be closely related to the previous chapter while focusing on the following entities and themes:

    
    Time Period: {time_period}
    Themes: {themes}
    Entity: {entity}

    Previous content: {prev_content}

    {format_instructions}
    """


def make_chapter(
    theme: str,
    entity: List[str],
    time: List[str],
    prev_content: str,
    model_name: str = "gpt-4",
    temperature: int = 0,
) -> Chapter:
    chapter_parser = PydanticOutputParser(pydantic_object=Chapter)

    print("Generating chapter...")
    print(f"Theme: {theme}, type: {type(theme)}")
    print(f"Entity: {entity}, type: {type(entity)}")
    print(f"Time: {time}, type: {type(time)}")
    print(f"Prev content: {prev_content}, type: {type(prev_content)}")

    # Create a prompt template
    formatted_prompt = PromptTemplate(
        template=generate_chapter_prompt(
            time_period=time,
            themes=theme,
            entity=entity,
            prev_content=prev_content,
            format_instructions=chapter_parser.get_format_instructions(),
        )
    )

    model = OpenAI(model_name=model_name, temperature=temperature)
    output = model(formatted_prompt)
    return chapter_parser.parse(output)


def create_story(alignment):
    story = Story(chapters=[])
    prev_content = ""

    print("Generating story...")

    for node, details in alignment.items():
        print(f"Generating chapter {node}...")

        new_chapter = make_chapter(
            details["Theme"], details["Entity"], details["Time"], prev_content
        )

        print(new_chapter)
        story.chapters.append(new_chapter)
        prev_content = new_chapter

    return story

In [6]:
story = create_story(alignment_results)

Generating story...
Generating chapter A1...
Generating chapter...
Theme: Space & Humanity: The human aspiration to explore and inhabit space., type: <class 'str'>
Entity: ['Hannah Reid: An advocate for mental health awareness and support.', 'Helix Labs: At the forefront of genetic research and innovation.', 'Science Fair Extravaganza: A festival celebrating innovations and discoveries.', "Lunar Mission Launch: An event marking mankind's leap into space exploration.", 'Professor Ivan Chen: A pioneer in space exploration technologies.'], type: <class 'list'>
Time: ['2019-01-01', '2020-08-02'], type: <class 'list'>
Prev content: , type: <class 'str'>


KeyError: 'input_variables'